In [ ]:
!pip install "jedi>=0.16"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.0 MB/s eta 0:00:00


In [ ]:
!pip -q install -U vllm datasets transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 146.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/53

In [ ]:
import re
import math
import json
from typing import List, Dict, Any

from datasets import load_dataset
from vllm import LLM, SamplingParams

In [ ]:
MODEL_ID = "allenai/OLMo-2-1124-7B-SFT"
SPLIT = "test"
MAX_EXAMPLES = 200
K_LIST = [1, 4, 8, 16, 32, 64, 128]
N_SAMPLES = max(K_LIST)
TEMPERATURE = 0.6
TOP_P = 0.95
MAX_NEW_TOKENS = 512
BATCH_SIZE = 16
TENSOR_PARALLEL_SIZE = 1

In [ ]:
import os
import sys

# Patch the ipykernel file on disk so spawned worker processes inherit the fix (Gemini fix)
os.system("sed -i 's/raise io.UnsupportedOperation(\"fileno\")/return 1/g' /usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py")
# In-memory patch for the current process
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

# configure llm and parameters
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=True,
    max_model_len=2048,
    gpu_memory_utilization=0.90,
)

sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
)


INFO 05-03 23:27:41 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'allenai/OLMo-2-1124-7B-SFT'}


config.json:   0%|          | 0.00/675 [00:00<?, ?B/s]

INFO 05-03 23:28:01 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-03 23:28:01 [nixl_utils.py:34] NIXL is not available
WARNING 05-03 23:28:01 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-03 23:28:01 [model.py:555] Resolved architecture: Olmo2ForCausalLM
INFO 05-03 23:28:01 [model.py:1680] Using max model len 2048
INFO 05-03 23:28:01 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-03 23:28:01 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-03 23:28:01 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

(EngineCore pid=10461) INFO 05-03 23:28:02 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='allenai/OLMo-2-1124-7B-SFT', speculative_config=None, tokenizer='allenai/OLMo-2-1124-7B-SFT', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_tra

Loading pt checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(EngineCore pid=10461) INFO 05-03 23:28:56 [default_loader.py:384] Loading weights took 11.88 seconds
(EngineCore pid=10461) INFO 05-03 23:28:56 [gpu_model_runner.py:4879] Model loading took 13.6 GiB memory and 51.461077 seconds
(EngineCore pid=10461) INFO 05-03 23:29:06 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/9ecf0c5ede/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=10461) INFO 05-03 23:29:06 [backends.py:1128] Dynamo bytecode transform time: 9.02 s
(EngineCore pid=10461) INFO 05-03 23:29:13 [backends.py:376] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=10461) INFO 05-03 23:29:19 [backends.py:391] Compiling a graph for compile range (1, 8192) takes 12.76 s
(EngineCore pid=10461) INFO 05-03 23:29:23 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/df2e194571d6e5d68e7a2d2c278545d55a3caeb2d27f05a0918a35fa7a0f6561/rank_0_0/model
(EngineCore pid=10461) I

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 18.67it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 23.27it/s]


(EngineCore pid=10461) INFO 05-03 23:29:43 [gpu_model_runner.py:6133] Graph capturing finished in 5 secs, took 0.48 GiB
(EngineCore pid=10461) INFO 05-03 23:29:43 [gpu_worker.py:599] CUDA graph pool memory: 0.48 GiB (actual), 0.51 GiB (estimated), difference: 0.04 GiB (7.3%).
(EngineCore pid=10461) INFO 05-03 23:29:43 [core.py:299] init engine (profile, create kv cache, warmup model) took 46.52 s (compilation: 26.19 s)
(EngineCore pid=10461) INFO 05-03 23:29:44 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


In [ ]:
# formatted prompt that pull in each question
def build_prompt(question):
    return (
        "<|im_start|>system\n"
        "You are a helpful assistant.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{question}\n"
        "Please reason step by step, and put your final answer within\\boxed{{}}.<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

In [ ]:
_gold_re = re.compile(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)")

def normalize_number_str(s):
    s = s.strip().replace(",", "")
    if re.fullmatch(r"[-+]?\d+\.0+", s):
        s = s.split(".")[0]
    return s

# only required for GSM8K set without separate answer field
def extract_gold_answer(answer_text):
    m = _gold_re.search(answer_text)
    if m:
        return normalize_number_str(m.group(1))
    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", answer_text)
    return normalize_number_str(nums[-1]) if nums else ""

def extract_pred_answer(completion):
    m = re.search(r"\\boxed\{([^}]*)\}", completion)
    if m:
        inside = m.group(1)
        nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", inside)
        if nums:
            return normalize_number_str(nums[-1])

    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", completion)
    if nums:
        return normalize_number_str(nums[-1])

    return ""

In [ ]:
def estimate_pass_at_k(n: int, c: int, k: int) -> float:
    """
    Unbiased estimator:
    pass@k = 1 - C(n-c, k) / C(n, k)
    """
    if c <= 0:
        return 0.0
    if n - c < k:
        return 1.0
    def log_comb(a: int, b: int) -> float:
        return math.lgamma(a + 1) - math.lgamma(b + 1) - math.lgamma(a - b + 1)

    log_num = log_comb(n - c, k)
    log_den = log_comb(n, k)
    ratio = math.exp(log_num - log_den)
    return 1.0 - ratio

In [ ]:
ds = load_dataset("HuggingFaceH4/aime_2024", split="train")

if MAX_EXAMPLES is not None:
    ds = ds.select(range(min(MAX_EXAMPLES, len(ds))))

examples = []
for ex in ds:
    examples.append({
        "question": ex["problem"],
        "gold": extract_gold_answer(ex["answer"]),
        "prompt": build_prompt(ex["problem"]),
    })

print(f"Loaded {len(examples)} Math 500 examples.")


Loaded 30 Math 500 examples.


In [ ]:
all_results: List[Dict[str, Any]] = []

for start in range(0, len(examples), BATCH_SIZE):
    batch = examples[start:start + BATCH_SIZE]
    prompts = [x["prompt"] for x in batch]

    outputs = llm.generate(prompts, sampling_params)

    for ex, out in zip(batch, outputs):
        preds = []
        correct_flags = []

        for cand in out.outputs:
            text = cand.text
            pred = extract_pred_answer(text)
            is_correct = (pred != "" and pred == ex["gold"])

            preds.append({
                "text": text,
                "pred": pred,
                "correct": is_correct,
            })
            correct_flags.append(is_correct)

        c = sum(correct_flags)

        row = {
            "question": ex["question"],
            "gold": ex["gold"],
            "num_correct": c,
            "samples": preds,
        }

        for k in K_LIST:
            row[f"pass@{k}"] = estimate_pass_at_k(N_SAMPLES, c, k)

        all_results.append(row)

    print(f"Processed {min(start + BATCH_SIZE, len(examples))}/{len(examples)}")


Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2048 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Processed 16/30


Rendering prompts:   0%|          | 0/14 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1792 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Processed 30/30


In [ ]:
metrics = {}
for k in K_LIST:
    metrics[f"pass@{k}"] = sum(r[f"pass@{k}"] for r in all_results) / len(all_results)

print("\n=== Final Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


with open("base_aime2024_passk_results.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": MODEL_ID,
            "num_examples": len(all_results),
            "n_samples": N_SAMPLES,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": MAX_NEW_TOKENS,
            "metrics": metrics,
            "results": all_results,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved detailed results to base_aime2024_passk_results.json")


=== Final Metrics ===
pass@1: 0.0003
pass@4: 0.0010
pass@8: 0.0021
pass@16: 0.0042
pass@32: 0.0083
pass@64: 0.0167
pass@128: 0.0333

Saved detailed results to base_aime2024_passk_results.json
